In [2]:
import os
from pathlib import Path

if Path.cwd().name == "examples":
    os.chdir(Path.cwd().parent)

from hybrid_rag import providers
from hybrid_rag.config import settings

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,  # 'force=True' penting di Jupyter agar menimpa handler root bawaan IPython
)


In [4]:
import asyncio
import functools
import logging
import math
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import httpx
import openai
from tenacity import before_sleep_log, retry, retry_if_exception, stop_after_attempt

logger = logging.getLogger("host.models")


def is_transient(exc: BaseException) -> bool:
    """True kalau mengirim ulang permintaan yang sama bisa berhasil."""
    if isinstance(exc, httpx.HTTPStatusError):  # jalur rerank
        return exc.response.status_code in (408, 429) or exc.response.status_code >= 500
    if isinstance(exc, (httpx.TransportError, TimeoutError)):  # TimeoutError: stream macet di langchain-openai
        return True
    if isinstance(exc, (openai.APIConnectionError, openai.APITimeoutError)):
        return True
    return isinstance(exc, openai.APIStatusError) and (exc.status_code in (408, 409, 429) or exc.status_code >= 500)

def seconds(value) -> float | None:
    try:
        delay = float(value)
    except (TypeError, ValueError):
        try:
            delay = (parsedate_to_datetime(str(value)).astimezone(timezone.utc) - datetime.now(timezone.utc)).total_seconds()
        except (TypeError, ValueError, IndexError, OverflowError):
            return None
    return max(0.0, float(math.ceil(delay)))


def wait_seconds(state) -> float:
    """`retry_after` dari provider (body JSON, lalu header) kalau ada; kalau tidak 1s, 2s, 4s, maksimal 8s."""
    exc = state.outcome.exception()
    body = getattr(exc, "body", None)
    headers = getattr(getattr(exc, "response", None), "headers", None)
    delay = seconds(body.get("retry_after")) if isinstance(body, dict) else None
    if delay is None and headers is not None:
        delay = seconds(headers.get("retry-after"))
    return delay if delay is not None else min(8.0, 2.0 ** (state.attempt_number - 1))

def with_retry(attempts):
    def decorate(func):
        def policy(*args):
            return retry(
                stop=stop_after_attempt(max(1, attempts(*args))),
                wait=wait_seconds,
                retry=retry_if_exception(is_transient),
                before_sleep=before_sleep_log(logger, logging.WARNING),
                reraise=True,
            )(func)

        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def awrapper(*args, **kwargs):
                return await policy(*args)(*args, **kwargs)
            return awrapper

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return policy(*args)(*args, **kwargs)
        return wrapper

    return decorate

per_instance = with_retry(lambda self, *_: self.retry_attempts)

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

class RetryingChat(ChatOpenAI):
    retry_attempts: int = 1
    @per_instance
    def _generate_with_cache(self, *args, **kwargs):
        return super()._generate_with_cache(*args, **kwargs)
    @per_instance
    async def _agenerate_with_cache(self, *args, **kwargs):
        return await super()._agenerate_with_cache(*args, **kwargs)

class RetryingEmbeddings(OpenAIEmbeddings):
    retry_attempts: int = 1
    # Vectors paired by `index`, not position, on both paths (see providers.embed_by_index).
    @per_instance
    def embed_documents(self, texts, chunk_size=None, **kwargs):
        self._ensure_sync_client_available()
        create = functools.partial(self.client.create, **self._invocation_params, **kwargs)
        return providers.embed_by_index_sync(create, texts, chunk_size or self.chunk_size)
    @per_instance
    async def aembed_documents(self, texts, chunk_size=None, **kwargs):
        create = functools.partial(self.async_client.create, **self._invocation_params, **kwargs)
        return await providers.embed_by_index(create, texts, chunk_size or self.chunk_size)
    @per_instance
    def embed_query(self, *args, **kwargs):
        return super().embed_query(*args, **kwargs)
    @per_instance
    async def aembed_query(self, *args, **kwargs):
        return await super().aembed_query(*args, **kwargs)


In [6]:

def get_llm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens, **model_kwargs):
    return RetryingChat(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_tokens=max_tokens,
        max_retries=0, retry_attempts=max_retries, **model_kwargs,
    )

def get_vlm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens=None):
    return get_llm(model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_retries=max_retries, max_tokens=max_tokens)

def get_embeddings(model_id, *, base_url, api_key, timeout, max_retries):
    return RetryingEmbeddings(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout,
        max_retries=0, retry_attempts=max_retries,
        model_kwargs={"encoding_format": "float"},
        check_embedding_ctx_length=False,
    )

def get_reranker(model_id, *, base_url, api_key, timeout, max_retries):
    @with_retry(lambda *_: max_retries)
    async def post(payload):
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.post(base_url, json=payload, headers={"Authorization": f"Bearer {api_key}"})
            response.raise_for_status()
            return response.json()

    async def rerank(query, documents, top_n=None):
        if top_n is not None:
            top_n = min(top_n, len(documents))
        data = await post({"model": model_id, "query": query, "documents": documents, "top_n": top_n})
        return data["results"]

    return rerank


# Daftarkan ke hybrid_rag

In [7]:
providers.configure(get_llm=get_llm, get_vlm=get_vlm, get_embeddings=get_embeddings, get_reranker=get_reranker)

# LLM

In [8]:
settings.require("extract")
llm = providers.get_llm(
    settings.extract_llm_model,
    base_url=settings.extract_llm_binding_host,
    api_key=settings.extract_llm_binding_api_key,
    timeout=settings.extract_llm_timeout,
    temperature=0.0,
)

# VLM

In [9]:
import base64
from langchain_core.messages import HumanMessage

settings.require("vlm")
vlm = providers.get_vlm(
    settings.vlm_model,
    base_url=settings.vlm_binding_host,
    api_key=settings.vlm_binding_api_key,
    timeout=settings.vlm_timeout,
)
# image = base64.b64encode(Path("examples/contoh.png").read_bytes()).decode()  # ganti dengan gambar Anda
# message = HumanMessage(content=[
#     {"type": "text", "text": "Jelaskan gambar ini."},
#     {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image}"}},
# ])
# (await vlm.ainvoke([message])).content

# EMBEDDINGS

In [10]:
settings.require("embedding")
embeddings = providers.get_embeddings(
    settings.embedding_model,
    base_url=settings.embedding_binding_host,
    api_key=settings.embedding_binding_api_key,
    timeout=settings.embedding_timeout,
)
# vectors = await embeddings.aembed_documents(["pasal 1", "pasal 2"])
# len(vectors), len(vectors[0])  # (2, EMBEDDING_DIM)

# RERANK

In [11]:
rerank = providers.get_reranker(
    settings.rerank_model,
    base_url=settings.rerank_binding_host,
    api_key=settings.rerank_binding_api_key,
    timeout=settings.rerank_timeout,
)
# await rerank(query="ibu kota Indonesia", documents=["Jakarta adalah ibu kota.", "Pisang itu buah."], top_n=2)

# Ingestion

Skema database dibuat sekali lewat `make db-init`. Lalu parse file dan tulis ke satu workspace.

## Parameter `insert_content_list`

| Parameter | Tipe | Default | Fungsi |
| --- | --- | --- | --- |
| `content_list` | `list[dict]` | wajib | Blok hasil `parse_file` (teks, gambar, tabel, persamaan) |
| `file_path` | `str` / `Path` | `"unknown_document"` | Nama file untuk sitasi |
| `doc_id` | `str` | dibuat dari isi dokumen | ID dokumen; isi yang sama menghasilkan ID yang sama |
| `ontology` | `str` | `None` | File di `knowledge_graph/ontology/` (`.ttl` / `.owl`). Ekstraksi dipandu dan divalidasi skemanya. `None` memakai kontrak bawaan tanpa validasi. File yang tidak ada langsung error |
| `entity_registry` | `str` | `None` | File di `knowledge_graph/entity_registry/` (`.yaml`) berisi alias dan pasangan `never_merge`. `None` hanya memakai aturan deterministik. File yang tidak ada langsung error |
| `entity_resolution` | `bool` | `True` | Gabungkan varian nama entity dokumen ini sebelum ditulis ke graph. `False` memakai merge nama persis, dan `entity_registry` diabaikan |
| `entity_resolution_llm` | `bool` | `True` | LLM ikut menilai pasangan nama yang tidak bisa diputuskan aturan. Hanya berlaku bila `entity_resolution=True`; kalau panggilan LLM gagal, hasil aturan tetap dipakai |
| `orphan_repair` | `bool` | `True` | Untuk chunk yang punya entity tanpa relasi, satu panggilan LLM tambahan mencari relasi yang memang tertulis di teks. Biaya hanya di chunk tersebut |
| `prune_isolated` | `bool` | `True` | Di akhir batch, hapus entity dokumen ini yang tetap tanpa relasi (node image/table/equation dikecualikan). Teksnya tetap bisa dicari lewat chunk |
| `save_parent` | `bool` | `True` | Simpan parent chunk di samping child yang di-embed, supaya retrieval bisa mengembalikan passage utuh (`return_parent`). Matikan hanya untuk dokumen yang tidak akan jadi bukti |

- Pengaturan dari `ontology` sampai `save_parent` disimpan di metadata dokumen, jadi retry atau resume dokumen tetap memakai nilai aslinya.
- Parameter berlaku per dokumen, kecuali `prune_isolated`: penghapusan berjalan sekali di akhir batch, dan hanya mencakup dokumen yang diingest dengan `prune_isolated=True`.
- Ukuran chunk dan konkurensi bukan parameter pemanggilan, melainkan ENV: `PARENT_CHUNK_SIZE`, `PARENT_CHUNK_OVERLAP`, `CHILD_CHUNK_SIZE`, `CHILD_CHUNK_OVERLAP`, `MAX_GLEANING`, `MAX_EXTRACT_INPUT_TOKENS`, `MAX_PARALLEL_INGESTION`, `EMBEDDING_BATCH_NUM`.
- `parse_file(file_path)` hanya menerima path file dan mengembalikan `content_list`.

In [15]:
from hybrid_rag.ingestion import parse_file, workspace_pool as write_pool
from pathlib import Path
path = Path("examples/overview.pdf")  
async with parse_file(path) as content_list:
    async with write_pool.lease("workspace_j") as rag:
        await rag.insert_content_list(
            content_list,
            file_path=path,
            doc_id="doc-13",
            ontology=None,
            entity_registry="iccn",
            entity_resolution=True,
            entity_resolution_llm=True,
            orphan_repair=True,
            prune_isolated=True,
            save_parent=True,
        )

2026-10-18 18:02:50,121 [INFO] hybrid_rag.ingestion.parser.router: [parse] starting file=overview.pdf
2026-10-18 18:02:50,163 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Concurrent processing dengan 5 workers (10 halaman)


[Textract] Mengecek DPI efektif setiap halaman (threshold=1)...
[Textract] Semua 10 halaman lolos cek DPI efektif.


2026-10-18 18:02:53,836 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 2/10 selesai
2026-10-18 18:02:53,880 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 1/10 selesai
2026-10-18 18:02:56,353 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 4/10 selesai
2026-10-18 18:02:57,092 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 5/10 selesai
2026-10-18 18:02:57,277 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 7/10 selesai
2026-10-18 18:02:57,766 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 3/10 selesai
2026-10-18 18:02:59,167 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 6/10 selesai
2026-10-18 18:03:01,537 [INFO] hybrid_rag.ingestion.parser.pdf.amazon_textract.parse: [Textract] Halaman 8/10 selesai
2026-10-18 18:03:02,550 [INFO] hybrid_rag.ingestion.pars

[Textract] Formatted JSON disimpan ke: /tmp/rag_assets_o7uqctoh/formatted_parsing_result.json


2026-10-18 18:03:09,894 [INFO] hybrid_rag.ingestion.pipeline: Processing 1 document(s)
2026-10-18 18:03:11,666 [INFO] hybrid_rag.ingestion.pipeline: Parsing: doc-13
2026-10-18 18:03:15,654 [INFO] hybrid_rag.ingestion.pipeline: Extracting stage 1/1: overview.pdf
2026-10-18 18:03:15,656 [INFO] hybrid_rag.ingestion.pipeline: Processing d-id: doc-13
2026-10-18 18:03:20,418 [INFO] hybrid_rag.ingestion.pipeline: Chunking parent-child, doc_id: doc-13
2026-10-18 18:03:20,501 [INFO] hybrid_rag.ingestion.chunking.parent_child: [parent-child] Split into 4 parents, 36 children (parent_size=2500, child_size=350)
2026-10-18 18:03:21,664 [INFO] hybrid_rag.ingestion.chunking.parent_child: [parent-child] Contextualizing 36 children (concurrency=2)...
2026-10-18 18:03:22,876 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-18 18:03:23,745 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/embeddings "HTTP/1.1 200 OK"
2026-10-18 18:

# Retrieval

In [13]:
from hybrid_rag.retrieval import QueryParam, retrieve_context

result = await retrieve_context(
    "Apa itu rindekraf",
    workspace="workspace_g",
    param=QueryParam(include_citations=True),
)
print(result.context)

2026-10-16 14:07:41,642 [INFO] hybrid_rag.workspace_pool: [WorkspacePool] initializing instance for workspace: workspace_g
2026-10-16 14:07:41,666 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_entities_mjai_titan_embed_text_v2_0_1024d
2026-10-16 14:07:41,668 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_relations_mjai_titan_embed_text_v2_0_1024d
2026-10-16 14:07:41,669 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_chunks_mjai_titan_embed_text_v2_0_1024d
2026-10-16 14:07:41,672 [INFO] hybrid_rag.storage.postgres.connection: Database : iccn_data_lake (retrieval pool)
2026-10-16 14:07:41,674 [INFO] hybrid_rag.storage.postgres.connection: PostgreSQL, Retry config: attempts=10, backoff=3.0s, backoff_max=30.0s, pool_close_timeout=7.0s
2026-10-16 14:07:41,676 [INFO] hybrid_rag.storage.postgres.connection: PostgreSQL, SSL mode set to: disable
2026-10-16 14:07:41,677 [INFO] hybrid_rag.storage.postgres.conn

<context>
<knowledge_graph>
Entities :
- **Rindekraf** (program): Rencana Induk Ekonomi Kreatif yang ditinjau kembali untuk menetapkan perubahan pengelompokan subsektor ekonomi kreatif sesuai perkembangan ilmu pengetahuan, teknologi, dan dinamika industri.
- **Rencana Induk Ekonomi Kreatif** (program): Rencana Induk Ekonomi Kreatif, yang disingkat Rindekraf, adalah dokumen perencanaan strategis nasional Indonesia untuk pengembangan sektor ekonomi kreatif periode 2026-2045. Rencana ini ditetapkan melalui Peraturan Presiden Nomor 37 Tahun 2026 dan menjadi acuan utama pengembangan ekonomi kreatif Indonesia yang diintegrasikan dalam dokumen perencanaan kementerian, lembaga, dan Pemerintah Daerah. Rindekraf memuat prinsip, visi, misi, tujuan, ruang lingkup, arah kebijakan, sasaran, strategi, dan pemangku kepentingan dengan pendekatan rantai nilai komprehensif. Dokumen ini menyusun pembaruan struktur ruang lingkup usaha melalui pendekatan klasifikasi subsektor ke dalam klaster. Terdapat dela

# Generation

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from hybrid_rag.retrieval import QueryParam, retrieve_context

async def get_context(question: str) -> str:
    result = await retrieve_context(
        question,
        workspace="workspace_e",
        param=QueryParam(include_citations=True),
    )
    return result.context

prompt = ChatPromptTemplate.from_messages([
    ("system", """<role>
You are a thorough and informative AI assistant.
Answer the user's question based only on the context provided below.
</role>"""),
    (
        "human",
        """<context>
{context}
</context>

Question: {question}""",
    ),
])

chain = (
    {
        "context": RunnableLambda(get_context), 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "Gimana timeline Penyusunan Outcome WCCE nya"
response = await chain.ainvoke(question)

print(response)

2026-10-13 16:46:04,247 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 16:46:04,313 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: mix:keywords:56bd6b71f8feb0f03b5b966a862b5e82
2026-10-13 16:46:05,594 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/embeddings "HTTP/1.1 200 OK"
2026-10-13 16:46:05,785 [INFO] hybrid_rag.retrieval.search: Query nodes: WCCE (top_k:5, cosine:0.15)
2026-10-13 16:46:06,593 [INFO] hybrid_rag.retrieval.search: Local query: 5 entites, 15 relations
2026-10-13 16:46:06,595 [INFO] hybrid_rag.retrieval.search: Query edges: timeline penyusunan, outcome (top_k:5, cosine:0.15)
2026-10-13 16:46:06,839 [INFO] hybrid_rag.retrieval.search: Global query: 8 entites, 5 relations
2026-10-13 16:46:07,213 [INFO] hybrid_rag.retrieval.search: Naive query: 46 chunks (chunk_top_k:10 cosine:0.15)
2026-10-13 16:46:07,215 [INFO] hybrid_rag.retrieval.search: Raw search results: 13 entities, 20 relat

Berdasarkan konteks yang disediakan, berikut adalah **Timeline Penyusunan Outcome World Conference on Creative Economy (WCCE) 2026**:

**1. FCE Meeting 2026 (Juli – September 2026)**
*   **Tujuan:** Menyusun *Jakarta Vision on Creative Economy* secara kolaboratif.
*   **Pelaksanaan:**
    *   Tahap pertama dilaksanakan secara **virtual** pada periode Juli hingga September 2026 (1x per bulan).
    *   Tahap kedua dilaksanakan secara **offline** saat acara WCCE 2026 berlangsung.
*   **Penanggung Jawab:** Dilaksanakan bersama oleh Kementerian Ekonomi Kreatif (Ekraf) dan Kementerian Luar Negeri (Kemlu) [1][2].

**2. Ministerial Meeting (21 Oktober 2026)**
*   **Kegiatan:** Pertemuan tingkat menteri untuk adopsi *Jakarta Vision on Creative Economy*.
*   **Penanggung Jawab:** Dilaksanakan oleh Kementerian Ekonomi Kreatif [1][5].

**3. Adopsi Jakarta Vision (22 Oktober 2026)**
*   **Output:** Penetapan *Jakarta Vision on Creative Economy* sebagai dokumen hasil utama WCCE 2026 [1][5].

**4. Pr

# Deletion

In [ ]:
from hybrid_rag.ingestion import delete_document

result = await delete_document("doc-2", workspace="workspace_e")
print(result)